# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


C:\Users\A\AppData\Local\Temp\ipykernel_19008\2076300660.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\A\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_docs = []

    for pdf_file in Path(pdf_directory).glob('**/*.pdf'):
        try:
            documents = PyPDFLoader(str(pdf_file)).load()
        except Exception as e:
            print(f"Warning: failed to load '{pdf_file}': {e}")
            continue

        for doc in documents:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'

        all_docs.extend(documents)

    return all_docs
    pass


In [3]:
pdf_directory = "data/pdfs" 
all_pdf_documents = process_all_pdfs(pdf_directory)
print(f"Loaded {len(all_pdf_documents)} pages from PDFs")

Loaded 70 pages from PDFs


# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs
    pass


In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 70 documents into 129 chunks


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-03-17T07:02:37+00:00', 'moddate': '2026-05-03T14:05:58+05:30', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': 'data\\pdfs\\Tut2.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'Tut2.pdf', 'file_type': 'pdf'}, page_content='ESC112 Tutorial 2\nSECTION: INTRODUCTION\n•Pointers are special kinds of variables (or data types) that hold thememory location of\nanother variablerather than storing a value directly.\n•The “another variable” mentioned above can be of any type:int,unsigned char,long\ndouble, etc. Corresponding pointer declarations would be:\n1int *x;\n2unsigned char *y;\n3long double *z;\n•In a computer, every data item occupies one or more contiguous memory cells called abyte.\nTypical sizes are:\nType Typical Size\nchar 1 byte\nint 4 bytes\nfloat 4 bytes\ndouble 8 bytes\nMemory 

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        print(f"Loading embedding model: {self.model_name}")

        self.model = SentenceTransformer(self.model_name)
        embedding_dim = self.model.get_sentence_embedding_dimension()
        print(f"Model loaded successfully. Embedding dimension: {embedding_dim}")
        pass

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if self.model is None:
            raise ValueError("Model not loaded. Cannot generate embeddings.")

        embeddings = self.model.encode(texts, show_progress_bar=True)

        return embeddings
        pass



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
        )

        print(f"Vector store initialized. Collection '{self.collection_name}' "
        f"contains {self.collection.count()} documents.")
        pass

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):

            raise ValueError(
            f"Number of documents ({len(documents)}) does not match "
            f"number of embeddings ({len(embeddings)})"
            )

        ids = []
        embeddings_list = []
        metadatas = []
        documents_content = []

        for i, doc in enumerate(documents):
            content = doc.page_content
            ids.append(uuid.uuid4().hex[:8])
            documents_content.append(content)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(content)
            embeddings_list.append(embeddings[i].tolist())
            metadatas.append(metadata)
        
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_content
        )

        print(f"Added {len(documents)} documents to the vector store.")
        
        pass  


In [ ]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-03-17T07:02:37+00:00', 'moddate': '2026-05-03T14:05:58+05:30', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': 'data\\pdfs\\Tut2.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'Tut2.pdf', 'file_type': 'pdf'}, page_content='ESC112 Tutorial 2\nSECTION: INTRODUCTION\n•Pointers are special kinds of variables (or data types) that hold thememory location of\nanother variablerather than storing a value directly.\n•The “another variable” mentioned above can be of any type:int,unsigned char,long\ndouble, etc. Corresponding pointer declarations would be:\n1int *x;\n2unsigned char *y;\n3long double *z;\n•In a computer, every data item occupies one or more contiguous memory cells called abyte.\nTypical sizes are:\nType Typical Size\nchar 1 byte\nint 4 bytes\nfloat 4 bytes\ndouble 8 bytes\nMemory 

## convert text to embeddings


In [10]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['ESC112 Tutorial 2\nSECTION: INTRODUCTION\n•Pointers are special kinds of variables (or data types) that hold thememory location of\nanother variablerather than storing a value directly.\n•The “another variable” mentioned above can be of any type:int,unsigned char,long\ndouble, etc. Corresponding pointer declarations would be:\n1int *x;\n2unsigned char *y;\n3long double *z;\n•In a computer, every data item occupies one or more contiguous memory cells called abyte.\nTypical sizes are:\nType Typical Size\nchar 1 byte\nint 4 bytes\nfloat 4 bytes\ndouble 8 bytes\nMemory cells are arranged sequentially and are identified by continuoushexadecimal ad-\ndresses.\n•When we declare\n1double d;\nthe compiler automatically allocates 8 bytes of storage ford. The variable can also be referred\nto by its starting memory address.\nThe address of a variable can be obtained using the unary&(address-of) operator.\n1double d, *pd = &d;\nDo not confuse&with the bitwise AND operator. The compiler distingui

In [11]:
## Generate the Embeddings
embedding_manager = EmbeddingManager()
vectorstore = VectorStore()
embeddings=embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks,embeddings)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2574.51it/s]
C:\Users\A\AppData\Local\Temp\ipykernel_19008\1572793511.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = self.model.get_sentence_embedding_dimension()


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection 'pdf_documents' contains 129 documents.


Batches: 100%|██████████| 5/5 [00:12<00:00,  2.59s/it]


Added 129 documents to the vector store.


# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        query_embedding = self.embedding_manager.generate_embeddings(query)
        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )


        retrieved_docs = []
        ids = results['ids'][0]
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        print(f"DEBUG: Found {len(documents)} raw documents in ChromaDB.")
        print(f"DEBUG: Raw distances returned: {distances}")

        for rank, (doc_id, content, metadata, distance) in enumerate(
            zip(ids, documents, metadatas, distances), start=1
        ):
            similarity_score = 1 - distance

            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    'id': doc_id,
                    'content': content,
                    'metadata': metadata,
                    'similarity_score': similarity_score,
                    'distance': distance,
                    'rank': rank
                })

        return retrieved_docs
        pass


In [13]:
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)



In [32]:

results = rag_retriever.retrieve("what is dynamic memory allocation", top_k=3, score_threshold=-0.1)
for r in results:
    print(f"Rank {r['rank']} | Score: {r['similarity_score']:.4f}")
    print(r['content'])
    print("---")

Batches: 100%|██████████| 1/1 [00:00<00:00, 16.59it/s]

DEBUG: Found 3 raw documents in ChromaDB.
DEBUG: Raw distances returned: [0.8079143166542053, 0.8079143166542053, 0.8843145966529846]
Rank 1 | Score: 0.1921
ESC112 Tutorial 4
1 Introduction
Topics to be covere:
•dynamic memory allocation
•multi dimensional arrays
•pointer to pointers
•application to matrices and images
Codes etc files can be found in the hello IITK folder for tutorial 4 under resources.
2 Dynamic memory allocation
there are 4 functions in stdlib.h to help with memory allocation.
•
1void *calloc(size_t nobj, size_t size)
calloc returns a pointer to space for an array of nobj objects, each of size size, or NULL if
the request cannot be satisfied. The space is initialized to zero bytes.
•
1void *malloc(size_t size)
malloc returns a pointer to space for an object of size size, or NULL if the request cannot be
satisfied. The space is uninitialized.
•
1void *realloc(void *p, size_t size)
realloc changes the size of the object pointed to by p to size. The contents will be unc

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [40]:
def rag_simple(query,retriever,llm):
    """Simple RAG pipeline: retrieve relevant docs, build a prompt, and query the LLM"""

    retrieved_docs = retriever.retrieve(query, top_k=3, score_threshold=-0.1)


    context = "\n\n".join([doc['content'] for doc in retrieved_docs])

    prompt = f"""You are a helpful assistant. Use the following pieces of context to answer the question. 
    If the context is empty or you don't know the answer, say exactly 'The context is empty'.

    Context:
    {context}

    Question: {query}

    Answer:"""

    response = llm.invoke(prompt)

    return response.content
    pass


In [41]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

if "GROQ_API_KEY" in os.environ:
    
    print("Success: API Key loaded securely into memory.")
else:
    print("Error: API Key not found. Check your .env file.")

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.0
)

Success: API Key loaded securely into memory.


In [44]:
answer=rag_simple("what is dynamic memory allocation?",rag_retriever,llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 20.85it/s]

DEBUG: Found 3 raw documents in ChromaDB.
DEBUG: Raw distances returned: [0.8243404030799866, 0.8243404030799866, 0.9085785150527954]
Dynamic memory allocation refers to the process of assigning memory to a program at runtime, rather than at compile time. This is done using functions from the stdlib.h library, which includes calloc, malloc, and realloc. These functions allow a program to request a specific amount of memory, and if the request is satisfied, they return a pointer to the allocated memory space. 

In the context of the provided tutorial, dynamic memory allocation is one of the topics to be covered, along with multi-dimensional arrays, pointer to pointers, and application to matrices and images. The tutorial also explains the different functions used for dynamic memory allocation, including their parameters, return values, and effects on memory. 

For example, the calloc function allocates space for an array of objects, initializing the space to zero bytes, while the malloc

# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # TODO: Implement advanced RAG retrieval and generation.
    # 1. Retrieve top_k documents using score threshold min_score.
    # 2. Parse sources including source_file, page, similarity_score, and a content preview.
    # 3. Determine confidence (e.g. max similarity score of retrieved docs).
    # 4. Invoke LLM with formatted prompt combining context and query.
    # 5. Return dict containing 'answer', 'sources', 'confidence' (and 'context' if return_context is True).
    pass


In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # TODO: Implement the AdvancedRAGPipeline query logic:
        # 1. Retrieve documents using self.retriever.
        # 2. Parse sources/metadata and construct the context.
        # 3. If stream=True, simulate streaming by printing the prompt in chunks.
        # 4. Generate the answer using self.llm.
        # 5. Construct citations list and append it to the answer.
        # 6. If summarize=True, call the LLM to get a 2-sentence summary of the answer.
        # 7. Append the question, answer, sources, and summary to self.history.
        # 8. Return dictionary containing question, answer (with citations), sources, summary, and history.
        pass
